# Data Preparation — Balón: micro-interpolación Akima y canal de ceguera

Pipeline híbrido para el reto **More than a Hack 2027: Ball Possession**.

Decisiones implementadas:

- Alineación `video_frame = frame_id - 1`.
- Micro-interpolación con **Akima en NumPy puro** (sin `scipy`).
- `ball_is_missing = 1` **solo** cuando el balón sigue desaparecido tras la limpieza.
- Los huecos largos **nunca** se rellenan: quedan `NaN` y la red debe mirar a los jugadores.
- `ball_is_imputed` se guarda como diagnóstico, sin cambiar la semántica de `ball_is_missing`.

## 1. Configuración y rutas

`MAX_INTERP_GAP_FRAMES` es el único parámetro que hay que tocar para cambiar el umbral.

In [1]:
import os
import sys
import time
from dataclasses import dataclass

import numpy as np
import pandas as pd


def _find_repo_root(start):
    """Sube por el árbol hasta encontrar games.py."""
    current = os.path.abspath(start)
    while True:
        if os.path.isfile(os.path.join(current, "games.py")):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            raise RuntimeError(
                "No se encontró games.py. Ejecuta el notebook dentro del repositorio.")
        current = parent


CWD = os.path.abspath(os.getcwd())
ROOT = _find_repo_root(CWD)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from games import GAME, GAMES, require

FPS = 25
DT = 1.0 / FPS
N_FRAMES = GAMES[GAME]["n_video_frames"]
PITCH_LEN, PITCH_WID = GAMES[GAME]["pitch"]

# ---------------------------------------------------------------------------
# UMBRAL CONFIGURABLE
# 25 frames = 1 segundo a 25 fps.
# Cambiar aquí si se quiere otro comportamiento:
#   - 24 para exigir estrictamente "< 1 segundo".
#   - 50 para permitir huecos de hasta 2 segundos.
# ---------------------------------------------------------------------------
MAX_INTERP_GAP_FRAMES = 25

NOTEBOOK_DIR = (
    CWD if os.path.basename(CWD) == "akima_with_data_prep"
    else os.path.join(ROOT, "models", "akima_with_data_prep")
)
OUT_NPZ = os.path.join(NOTEBOOK_DIR, "ball_clean.npz")

print(f"Repositorio: {ROOT}")
print(f"Frames: {N_FRAMES:,} | FPS: {FPS} | Pitch: {PITCH_LEN} x {PITCH_WID} m")
print(f"Umbral de micro-interpolación: {MAX_INTERP_GAP_FRAMES} frames")

Repositorio: c:\Barcelona2026\more-than-a-hack-2027
Frames: 199,319 | FPS: 25 | Pitch: 105.3 x 68.0 m
Umbral de micro-interpolación: 25 frames


## 2. Carga eficiente del ATD

Cada línea del fichero tiene la forma `frame_id:jugadores:bx,by,velocidad`.
Se usa `find`/`rfind` para extraer solo el `frame_id` y el campo del balón, sin
materializar los 46 tracks de jugadores. La alineación con el envío es
`video_frame = frame_id - 1`.

In [3]:
def load_ball_track(raw_path, n_frames=N_FRAMES):
    """Carga solo la trayectoria del balón y la alinea con los frames del vídeo.

    Devuelve un DataFrame con video_frame, ball_x, ball_y y ball_speed_mps.
    Los frames sin balón quedan como NaN.
    """
    ball_x = np.full(n_frames, np.nan, dtype=np.float32)
    ball_y = np.full(n_frames, np.nan, dtype=np.float32)
    ball_speed = np.full(n_frames, np.nan, dtype=np.float32)

    n_records = 0
    first_id = None
    last_id = None

    with open(raw_path, "r") as handle:
        for line in handle:
            first_sep = line.find(":")
            last_sep = line.rfind(":")
            if first_sep < 0 or last_sep <= first_sep:
                continue

            frame_id = int(line[:first_sep])
            index = frame_id - 1
            if index < 0 or index >= n_frames:
                continue

            n_records += 1
            if first_id is None:
                first_id = frame_id
            last_id = frame_id

            ball = line[last_sep + 1:].strip()
            parts = ball.split(",")
            if len(parts) < 2:
                continue

            bx = float(parts[0])
            by = float(parts[1])
            if np.isfinite(bx) and np.isfinite(by):
                ball_x[index] = bx
                ball_y[index] = by
                if len(parts) > 2:
                    speed = float(parts[2])
                    if np.isfinite(speed):
                        ball_speed[index] = speed

    if n_records != n_frames or first_id != 1 or last_id != n_frames:
        raise ValueError(
            "ATD no contiguo: records={}, first={}, last={}, expected={}".format(
                n_records, first_id, last_id, n_frames))

    return pd.DataFrame({
        "video_frame": np.arange(n_frames, dtype=np.int64),
        "ball_x": ball_x,
        "ball_y": ball_y,
        "ball_speed_mps": ball_speed,
    })

## 3. Detección vectorizada de huecos

Se detectan todos los tramos de `NaN` en una sola operación con `np.diff`.
También se generan features temporales del hueco (`gap_len`, `gap_age`,
`gap_remaining`) que la TCN puede usar como contexto.

In [4]:
def find_gaps(observed):
    """Devuelve inicios y finales (exclusivos) de cada hueco del balón."""
    observed_i8 = observed.astype(np.int8)
    d = np.diff(np.concatenate(([1], observed_i8, [1])))
    starts = np.flatnonzero(d == -1)
    ends = np.flatnonzero(d == 1)
    return starts, ends


def build_gap_features(starts, ends, n_frames):
    """Construye features temporales del hueco para cada frame."""
    gap_lengths = ends - starts
    gap_len = np.zeros(n_frames, dtype=np.uint16)
    gap_age = np.zeros(n_frames, dtype=np.uint16)
    gap_remaining = np.zeros(n_frames, dtype=np.uint16)

    for start, end, length in zip(starts, ends, gap_lengths):
        gap_len[start:end] = length
        gap_age[start:end] = np.arange(length, dtype=np.uint16)
        gap_remaining[start:end] = np.arange(length - 1, -1, -1, dtype=np.uint16)

    return gap_lengths, gap_len, gap_age, gap_remaining

## 4. Akima en NumPy puro

- `akima_tangents` calcula las pendientes de Akima en los anclajes observados.
- `akima_evaluate` evalúa el polinomio de Hermite cúbico de forma vectorizada.
- Si hay menos de 2 anclajes se usa interpolación lineal como fallback.

In [5]:
def akima_tangents(x, y):
    """Pendientes de Akima para una serie x estrictamente creciente."""
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    n = len(x)

    if n < 2:
        raise ValueError("Akima necesita al menos 2 puntos")

    h = np.diff(x)
    if np.any(h <= 0):
        raise ValueError("x debe ser estrictamente creciente")

    m = np.diff(y) / h
    if n == 2:
        return np.array([m[0], m[0]], dtype=np.float64)

    p = np.empty(n + 3, dtype=np.float64)
    p[2:-2] = m
    p[1] = 2.0 * m[0] - m[1]
    p[0] = 2.0 * p[1] - m[0]
    p[-2] = 2.0 * m[-1] - m[-2]
    p[-1] = 2.0 * p[-2] - m[-1]

    w1 = np.abs(p[3:] - p[2:-1])
    w2 = np.abs(p[1:-2] - p[:-3])
    denom = w1 + w2

    tangents = np.empty(n, dtype=np.float64)
    nonzero = denom > 0
    tangents[nonzero] = (
        w1[nonzero] * p[1:-2][nonzero] + w2[nonzero] * p[2:-1][nonzero]
    ) / denom[nonzero]
    tangents[~nonzero] = 0.5 * (p[1:-2][~nonzero] + p[2:-1][~nonzero])
    return tangents


def akima_evaluate(x, y, tangents, x_target):
    """Evalúa el spline de Akima en x_target usando Hermite cúbico."""
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    tangents = np.asarray(tangents, dtype=np.float64)
    x_target = np.asarray(x_target, dtype=np.float64)

    k = np.searchsorted(x, x_target, side="right") - 1
    k = np.clip(k, 0, len(x) - 2)

    dx = x[k + 1] - x[k]
    u = (x_target - x[k]) / dx

    h00 = 2.0 * u ** 3 - 3.0 * u ** 2 + 1.0
    h10 = u ** 3 - 2.0 * u ** 2 + u
    h01 = -2.0 * u ** 3 + 3.0 * u ** 2
    h11 = u ** 3 - u ** 2

    return (
        h00 * y[k]
        + h10 * dx * tangents[k]
        + h01 * y[k + 1]
        + h11 * dx * tangents[k + 1]
    )

## 5. Limpieza híbrida y canal de ceguera

`ball_is_missing` sigue exactamente la regla acordada:

- `0`: balón observado **o** hueco corto rellenado de forma segura.
- `1`: hueco largo; las coordenadas siguen `NaN` y la red debe ignorarlas.

`clip_to_pitch` se deja en `False` por defecto: las coordenadas observadas
pueden salir ligeramente de `[0, 1]` por ruido del tracker, y recortar solo los
frames imputados crearía discontinuidades en los bordes del hueco.

In [6]:
@dataclass
class CleanBallTrack:
    video_frame: np.ndarray
    ball_x: np.ndarray
    ball_y: np.ndarray
    ball_speed_mps: np.ndarray
    ball_observed: np.ndarray
    ball_is_imputed: np.ndarray
    ball_is_missing: np.ndarray
    ball_gap_len: np.ndarray
    ball_gap_age: np.ndarray
    ball_gap_remaining: np.ndarray
    last_seen_ball_x: np.ndarray
    last_seen_ball_y: np.ndarray
    starts: np.ndarray
    ends: np.ndarray
    gap_lengths: np.ndarray
    n_observed: int
    n_imputed: int
    n_missing: int


def clean_ball_track(ball_x, ball_y, ball_speed=None,
                     max_gap_frames=MAX_INTERP_GAP_FRAMES,
                     clip_to_pitch=False):
    """Aplica micro-interpolación Akima y crea el canal ball_is_missing."""
    ball_x = np.asarray(ball_x, dtype=np.float32).copy()
    ball_y = np.asarray(ball_y, dtype=np.float32).copy()
    n = len(ball_x)

    if ball_speed is None:
        ball_speed = np.full(n, np.nan, dtype=np.float32)
    else:
        ball_speed = np.asarray(ball_speed, dtype=np.float32).copy()

    observed = np.isfinite(ball_x) & np.isfinite(ball_y)
    starts, ends = find_gaps(observed)
    gap_lengths, gap_len, gap_age, gap_remaining = build_gap_features(
        starts, ends, n)

    interpolable = (
        (gap_lengths <= max_gap_frames)
        & (starts > 0)
        & (ends < n)
    )

    imputed = np.zeros(n, dtype=np.uint8)
    anchor_idx = np.flatnonzero(observed)

    if anchor_idx.size >= 2:
        target_parts = [
            np.arange(start, end, dtype=np.int64)
            for start, end, take in zip(starts, ends, interpolable)
            if take
        ]
        if target_parts:
            target = np.concatenate(target_parts)
            anchor_x = anchor_idx.astype(np.float64)
            tangent_x = akima_tangents(
                anchor_x, ball_x[anchor_idx].astype(np.float64))
            tangent_y = akima_tangents(
                anchor_x, ball_y[anchor_idx].astype(np.float64))

            filled_x = akima_evaluate(
                anchor_x, ball_x[anchor_idx].astype(np.float64),
                tangent_x, target)
            filled_y = akima_evaluate(
                anchor_x, ball_y[anchor_idx].astype(np.float64),
                tangent_y, target)

            if clip_to_pitch:
                filled_x = np.clip(filled_x, 0.0, 1.0)
                filled_y = np.clip(filled_y, 0.0, 1.0)

            ball_x[target] = filled_x.astype(np.float32)
            ball_y[target] = filled_y.astype(np.float32)
            imputed[target] = 1

    missing = (~observed) & (imputed == 0)

    speed = ball_speed.copy()
    if imputed.any():
        idx = np.flatnonzero(imputed)
        left = np.maximum(idx - 1, 0)
        right = np.minimum(idx + 1, n - 1)
        dx_m = (
            ball_x[right].astype(np.float64)
            - ball_x[left].astype(np.float64)
        ) * PITCH_LEN
        dy_m = (
            ball_y[right].astype(np.float64)
            - ball_y[left].astype(np.float64)
        ) * PITCH_WID
        dt = (right - left).astype(np.float64) * DT
        speed[idx] = (np.hypot(dx_m, dy_m) / dt).astype(np.float32)
    speed[missing] = np.nan

    last_idx = np.where(observed, np.arange(n), -1)
    last_idx = np.maximum.accumulate(last_idx)
    last_x = np.full(n, np.nan, dtype=np.float32)
    last_y = np.full(n, np.nan, dtype=np.float32)
    valid = last_idx >= 0
    last_x[valid] = ball_x[last_idx[valid]]
    last_y[valid] = ball_y[last_idx[valid]]

    return CleanBallTrack(
        video_frame=np.arange(n, dtype=np.int64),
        ball_x=ball_x,
        ball_y=ball_y,
        ball_speed_mps=speed,
        ball_observed=observed.astype(np.uint8),
        ball_is_imputed=imputed,
        ball_is_missing=missing.astype(np.uint8),
        ball_gap_len=gap_len,
        ball_gap_age=gap_age,
        ball_gap_remaining=gap_remaining,
        last_seen_ball_x=last_x,
        last_seen_ball_y=last_y,
        starts=starts,
        ends=ends,
        gap_lengths=gap_lengths,
        n_observed=int(observed.sum()),
        n_imputed=int(imputed.sum()),
        n_missing=int(missing.sum()),
    )

## 6. Tests sintéticos

Cubren los casos límite del umbral y de los huecos sin anclaje: 24, 25 y 26
frames, hueco inicial, hueco final y serie completamente vacía.

In [7]:
def run_synthetic_tests():
    """Valida Akima y la regla de micro-interpolación con series sintéticas."""
    x = np.arange(20, dtype=np.float64)
    y = np.sin(x)
    tangents = akima_tangents(x, y)
    evaluated = akima_evaluate(x, y, tangents, x)
    assert np.allclose(evaluated, y, atol=1e-9), \
        "Akima no reproduce los puntos observados"

    def track_from_missing(length, missing_slice):
        bx = np.linspace(0.2, 0.8, length).astype(np.float32)
        by = np.linspace(0.3, 0.7, length).astype(np.float32)
        bx[missing_slice] = np.nan
        by[missing_slice] = np.nan
        return clean_ball_track(bx, by, clip_to_pitch=False)

    result = track_from_missing(100, slice(20, 44))
    assert result.ball_is_imputed[20:44].all()
    assert not result.ball_is_missing[20:44].any()
    assert np.isfinite(result.ball_x[20:44]).all()

    result = track_from_missing(100, slice(20, 45))
    assert result.ball_is_imputed[20:45].all()
    assert not result.ball_is_missing[20:45].any()
    assert np.isfinite(result.ball_x[20:45]).all()

    result = track_from_missing(100, slice(20, 46))
    assert not result.ball_is_imputed[20:46].any()
    assert result.ball_is_missing[20:46].all()
    assert np.isnan(result.ball_x[20:46]).all()

    result = track_from_missing(100, slice(0, 10))
    assert result.ball_is_missing[:10].all()
    assert np.isnan(result.ball_x[:10]).all()

    result = track_from_missing(100, slice(90, 100))
    assert result.ball_is_missing[90:].all()
    assert np.isnan(result.ball_x[90:]).all()

    bx = np.full(50, np.nan, dtype=np.float32)
    by = np.full(50, np.nan, dtype=np.float32)
    result = clean_ball_track(bx, by)
    assert result.ball_is_missing.all()
    assert np.isnan(result.ball_x).all()

    result = track_from_missing(100, slice(50, 51))
    assert result.ball_is_imputed[50]
    assert not result.ball_is_missing[50]
    assert np.isfinite(result.ball_x[50])
    assert np.isfinite(result.ball_speed_mps[50])

    print("Tests sintéticos: OK")


run_synthetic_tests()

Tests sintéticos: OK


## 7. Ejecución sobre el partido real

Carga el ATD táctico completo (199.319 frames), aplica la limpieza y verifica
las invariantes del canal de ceguera.

In [8]:
raw_path = require(["atd_raw"])[0]

t0 = time.perf_counter()
raw_df = load_ball_track(raw_path)
t_load = time.perf_counter() - t0

t0 = time.perf_counter()
track = clean_ball_track(
    raw_df["ball_x"].to_numpy(),
    raw_df["ball_y"].to_numpy(),
    raw_df["ball_speed_mps"].to_numpy(),
)
t_clean = time.perf_counter() - t0

observed = track.ball_observed.astype(bool)
imputed = track.ball_is_imputed.astype(bool)
missing = track.ball_is_missing.astype(bool)

assert len(track.video_frame) == N_FRAMES
assert np.array_equal(track.video_frame, np.arange(N_FRAMES))
assert np.isfinite(track.ball_x[observed]).all()
assert np.isfinite(track.ball_x[imputed]).all()
assert np.isnan(track.ball_x[missing]).all()
assert not (imputed & missing).any()
assert (observed | imputed | missing).all()
assert track.n_observed + track.n_imputed + track.n_missing == N_FRAMES

for start, end, length in zip(track.starts, track.ends, track.gap_lengths):
    if length > MAX_INTERP_GAP_FRAMES:
        assert np.isnan(track.ball_x[start:end]).all()
    elif start > 0 and end < N_FRAMES:
        assert np.isfinite(track.ball_x[start:end]).all()

summary = pd.DataFrame([
    ("Frames totales", N_FRAMES, 100.0),
    ("Observados", track.n_observed, 100.0 * track.n_observed / N_FRAMES),
    ("Micro-interpolados", track.n_imputed, 100.0 * track.n_imputed / N_FRAMES),
    ("Ciegos (ball_is_missing=1)", track.n_missing,
     100.0 * track.n_missing / N_FRAMES),
], columns=["Grupo", "Frames", "Porcentaje"])

gap_df = pd.DataFrame({
    "start_frame": track.starts,
    "end_frame": track.ends,
    "length_frames": track.gap_lengths,
    "length_seconds": track.gap_lengths / FPS,
    "interpolated": (
        (track.gap_lengths <= MAX_INTERP_GAP_FRAMES)
        & (track.starts > 0)
        & (track.ends < N_FRAMES)
    ),
})

print(f"Lectura ATD: {t_load:.2f} s")
print(f"Limpieza:    {t_clean:.2f} s")
print(f"Huecos totales: {len(track.gap_lengths):,}")
print(
    "  Interpolados (<= {} frames): {:,}".format(
        MAX_INTERP_GAP_FRAMES, int(gap_df["interpolated"].sum())))
print(
    "  Ciegos (> {} frames):       {:,}".format(
        MAX_INTERP_GAP_FRAMES,
        int((~gap_df["interpolated"]).sum())))
print()
display(summary)
display(gap_df.sort_values("length_frames", ascending=False).head(10))

Lectura ATD: 0.51 s
Limpieza:    0.02 s
Huecos totales: 186
  Interpolados (<= 25 frames): 43
  Ciegos (> 25 frames):       143



,Grupo,Frames,Porcentaje
0,Frames totales,199319,100.000000
1,Observados,113488,56.937873
2,Micro-interpolados,185,0.092816
3,Ciegos (ball_is_missing=1),85646,42.969311


,start_frame,end_frame,length_frames,length_seconds,interpolated
185,190630,199319,8689,347.56,False
149,157546,163083,5537,221.48,False
2,5108,10643,5535,221.40,False
0,0,4192,4192,167.68,False
42,58081,61547,3466,138.64,False
69,86615,88352,1737,69.48,False
57,73997,75532,1535,61.40,False
56,72560,73951,1391,55.64,False
31,46678,48031,1353,54.12,False
53,70196,71520,1324,52.96,False


## 8. Guardado del bundle

Se guardan las coordenadas limpias, el flag binario y las features temporales
del hueco. `last_seen_ball_x/y` quedan preparados para el cálculo posterior del
centroide 5NN.

In [9]:
np.savez_compressed(
    OUT_NPZ,
    video_frame=track.video_frame,
    ball_x=track.ball_x,
    ball_y=track.ball_y,
    ball_speed_mps=track.ball_speed_mps,
    ball_observed=track.ball_observed,
    ball_is_imputed=track.ball_is_imputed,
    ball_is_missing=track.ball_is_missing,
    ball_gap_len=track.ball_gap_len,
    ball_gap_age=track.ball_gap_age,
    ball_gap_remaining=track.ball_gap_remaining,
    last_seen_ball_x=track.last_seen_ball_x,
    last_seen_ball_y=track.last_seen_ball_y,
    gap_starts=track.starts,
    gap_ends=track.ends,
    gap_lengths=track.gap_lengths,
)

size_mb = os.path.getsize(OUT_NPZ) / (1024 * 1024)
print(f"Guardado: {OUT_NPZ} ({size_mb:.2f} MB)")
print("Claves: video_frame, ball_x, ball_y, ball_speed_mps,")
print("        ball_observed, ball_is_imputed, ball_is_missing,")
print("        ball_gap_len, ball_gap_age, ball_gap_remaining,")
print("        last_seen_ball_x, last_seen_ball_y, gap_starts,")
print("        gap_ends, gap_lengths")

Guardado: c:\Barcelona2026\more-than-a-hack-2027\models\akima_with_data_prep\ball_clean.npz (1.91 MB)
Claves: video_frame, ball_x, ball_y, ball_speed_mps,
        ball_observed, ball_is_imputed, ball_is_missing,
        ball_gap_len, ball_gap_age, ball_gap_remaining,
        last_seen_ball_x, last_seen_ball_y, gap_starts,
        gap_ends, gap_lengths


## 9. Notas de integración

- **TCN**: usar `ball_is_missing` como interruptor de atención. Cuando vale 1,
  las coordenadas del balón son `NaN`; la red debe apoyarse en los jugadores.
- **5NN**: `last_seen_ball_x/y` dan la última posición fiable antes del hueco.
  A partir de ahí se calcula el centroide de los 5 jugadores más cercanos.
- **Umbral**: modificar `MAX_INTERP_GAP_FRAMES` en la sección 1. El resto del
  pipeline se adapta automáticamente.
- **Huecos largos**: nunca se rellenan. `ball_is_missing = 1` y coordenadas
  `NaN`, tal como exige la estrategia híbrida.
- **Etiqueta X**: este notebook no decide la abstención. El flag alimenta al
  modelo; la política de `X` se aplica en post-proceso para proteger `spellF1`.